# 63 - Transposed Convolution (Upsampling)

**Section:** Convolution and transformations | **Prereqs:** `Convolution and transformations/Max_Mean_Pooling.ipynb` | **Next:** `Convolution and transformations/Image_transformations.ipynb`

Convolution and pooling make feature maps smaller. `ConvTranspose2d` makes them
*bigger*, with learned weights - the layer that lets a network generate an image
rather than classify one.

**Where it is used**
- **Autoencoders** - the decoder rebuilding an image from a small latent code
  (`Autoencoders/`).
- **GANs** - the generator turning a noise vector into a picture (`GANs/`).
- **Segmentation** - restoring a downsampled map to the input resolution.

**Its output size formula runs the other way:**

    out = (in - 1)*stride - 2*padding + kernel

So `stride=2` roughly doubles the size, mirroring how `stride=2` convolution
roughly halves it.

**It is not a deconvolution.** It does not invert a convolution; it applies the
same *pattern of connections* in reverse, with its own learned weights. The name
is historical and misleading.

**The checkerboard artifact:** when `kernel_size` is not divisible by `stride`,
output pixels receive uneven numbers of contributions and generated images show a
regular grid pattern. Use `kernel=4, stride=2`, or upsample-then-convolve
instead - a detail visible in some of the GAN outputs later in the course.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split

!pip install torchvision
import torchvision

In [8]:
# ConvTranspose2d with the same argument list as Conv2d.
# (1,3,64,64) -> (1,15,68,68): out = (64-1)*1 - 0 + 5 = 68.
# With stride=1 and a 5x5 kernel it GROWS by kernel-1 = 4, exactly the amount a
# forward convolution would have shrunk it.
img_size = (3,64,64)

input_chan = 3
ouput_chan = 15

stride = 1
padding = 0
kernal = 5

img = torch.rand(1,img_size[0],img_size[1],img_size[2])

c  = nn.ConvTranspose2d(in_channels=input_chan, out_channels= ouput_chan , stride=stride, padding=padding, kernel_size=kernal)

out = c(img)

In [10]:
# Shapes. Note the weight is (in_channels, out_channels, kH, kW) - the first two
# are SWAPPED relative to Conv2d, which is where the 'transpose' in the name
# comes from.
out.shape, c.weight.shape, c.bias.shape

(torch.Size([1, 15, 68, 68]), torch.Size([3, 15, 5, 5]), torch.Size([15]))

In [11]:
# The layer's configuration.
c

ConvTranspose2d(3, 15, kernel_size=(5, 5), stride=(1, 1))